<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Koch_Curve_Animation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualization of Fractal Geometry: The Koch Curve

This notebook contains the source code and configuration for rendering high-quality mathematical animations of the Koch Curve using the Manim library.

## Project Status

The project currently features a specialized **Sketchbook Aesthetic** designed for high-resolution mobile viewing. The latest iteration of the code simulates a fountain pen drawing on textured paper, providing an organic feel to the rigid mathematical precision of fractals.

### Key Features
* **Paper & Bleeding Ink Theme**: Uses a warm parchment background and deep indigo/burgundy ink palettes.
* **High Detail (Iteration 5)**: The algorithm now processes up to the 5th level of recursion, showcasing the extreme complexity of the fractal boundary.
* **Mobile Optimization**: Configured with a 9:16 aspect ratio (1080x1920) and large handwriting-style typography for maximum visibility on handheld devices.

## Mathematical Background

### The Koch Snowflake
The Koch Snowflake is one of the earliest described fractal curves, first appearing in a 1904 paper by Helge von Koch. It is constructed by starting with an equilateral triangle (or a single line segment for a Koch Curve) and recursively altering each segment.

### Construction Process
For every iteration (n), each line segment is divided into three equal parts:
1. The middle segment is removed.
2. Two new segments are added, forming an equilateral triangle pointing outwards where the middle segment used to be.
3. This process is repeated for each resulting segment in the next iteration.

### Mathematical Paradoxes
Fractals like the Koch Curve exhibit several fascinating properties:

* **Self-Similarity**: The curve looks identical regardless of the level of magnification.
* **Infinite Perimeter**: As the number of iterations approaches infinity, the total length of the curve also approaches infinity. This is governed by the formula $L_n = L_0 \cdot (4/3)^n$.
* **Finite Area**: Despite having an infinite boundary, a Koch Snowflake enclosed within a circle occupies a finite, constant area.

In [1]:
!apt-get update -qq
!apt-get install -y -qq libcairo2-dev libpango1.0-dev ffmpeg dvisvgm texlive-latex-extra texlive-fonts-extra
!pip install -q manim


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../000-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package fonts-lato.
Preparing to unpack .../001-fonts-lato_2.0-2.1_all.deb ...
Unpacking fonts-lato (2.0-2.1) ...
Selecting previously unselected package poppler-data.
Preparing to unpack .../002-poppler-data_0.4.11-1_all.deb ...
Unpacking poppler-data (0.4.11-1) ...
Selecting previously unselected package tex-common.
Preparing to unpack .../003-tex-common_6.17_all.deb ...
Unpacking tex-common (6.17) ...
Selecting previously unse

In [1]:
import manim
from manim.utils.ipython_magic import ManimMagic

try:
    # Manually register the %%manim magic command into the IPython shell
    get_ipython().register_magics(ManimMagic)
    print(f"Manim {manim.__version__} loaded and magic commands registered successfully.")
except Exception as e:
    print(f"Error loading Manim magic: {e}")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Manim 0.20.1 loaded and magic commands registered successfully.


In [ ]:
%%manim -v WARNING -r 1080,1920 --fps 30 --progress_bar None KochCurveAnimation

"""
Author: Mugambi Ndwiga
Optimized for Mobile Phone Visibility (High-Contrast, Large Typography)
Concept: The Koch Snowflake Geometry
"""

from manim import *
import numpy as np

class KochCurveAnimation(Scene):
    def construct(self):
        # --- Force 9:16 Aspect Ratio Coordinate Space ---
        config.frame_width = 9
        config.frame_height = 16

        # --- Configuration ---
        MAIN_COLOR = BLUE_D
        ACCENT_COLOR = YELLOW_C
        BG_COLOR = "#0a0a0a"
        self.camera.background_color = BG_COLOR

        # --- Watermark (Gently visible at the safe area boundary) ---
        watermark = Text(
            "© Mugambi Ndwiga / @craftsandengineering",
            font_size=18,
            fill_opacity=0.35
        ).move_to([0, -7.5, 0])
        self.add(watermark)

        # --- Zone 1: Title Block (Enlarged and bolded for mobile thumb-stopping power) ---
        title_main = Text("INFINITE PERIMETER", weight=BOLD, font_size=48)
        title_main.scale_to_fit_width(config.frame_width * 0.88)
        title_sub = Text("The Koch Curve", font_size=34, color=GRAY_A, weight=SEMIBOLD)
        title = VGroup(title_main, title_sub).arrange(DOWN, buff=0.3).move_to([0, 5.8, 0])

        # --- Zone 2: Static Formula Panel (Scaled up for small screen equation clarity) ---
        info_panel = MathTex(
            r"L_n = L_0 \cdot \left(\frac{4}{3}\right)^n",
            font_size=46
        ).move_to([0, 3.5, 0])

        # --- Zone 3: Koch Curve Base (Shifted slightly lower to allow vertical fractal expansion) ---
        start_pt = np.array([-3.2, -0.8, 0])
        end_pt = np.array([3.2, -0.8, 0])
        curve = Line(start_pt, end_pt, color=MAIN_COLOR, stroke_width=6)

        # --- Geometry Logic ---
        def get_koch_points(points, order):
            for _ in range(order):
                new_points = []
                for i in range(len(points) - 1):
                    p1 = points[i]
                    p2 = points[i+1]
                    v = p2 - p1
                    a = p1 + v / 3
                    b = p1 + 2 * v / 3
                    rotation_matrix = np.array([
                        [np.cos(PI/3), -np.sin(PI/3), 0],
                        [np.sin(PI/3), np.cos(PI/3), 0],
                        [0, 0, 1]
                    ])
                    peak = a + np.dot(rotation_matrix, v / 3)
                    new_points.extend([p1, a, peak, b])
                new_points.append(points[-1])
                points = new_points
            return points

        # --- Narrative Flow ---
        self.play(Write(title), run_time=1.5)
        self.play(FadeIn(info_panel, shift=DOWN * 0.2))
        self.play(Create(curve))
        self.wait(0.8)

        # --- Zone 4: Dynamic Length Updates (Thicker strokes and larger font math) ---
        displayed_length = None

        for n in range(1, 4):
            new_pts = get_koch_points([start_pt, end_pt], n)
            # Increased minimum stroke thickness to remain highly visible on compressed mobile feeds
            new_curve = VMobject(color=MAIN_COLOR, stroke_width=max(3.0, 6.5 - n))
            new_curve.set_points_as_corners(new_pts)

            next_length = MathTex(
                f"L_{n} = {round((4/3)**n, 2)} \\cdot L_0",
                font_size=42,
                color=ACCENT_COLOR
            ).move_to([0, -3.4, 0])

            if displayed_length is None:
                self.play(
                    Transform(curve, new_curve),
                    FadeIn(next_length, shift=UP * 0.2),
                    run_time=1.8
                )
                displayed_length = next_length
            else:
                self.play(
                    Transform(curve, new_curve),
                    Transform(displayed_length, next_length),
                    run_time=1.8
                )
            self.wait(1.2)

        # --- Zone 5: Conclusion Hook (Bold takeaway text) ---
        conclusion = Text(
            "As n → ∞, Length → ∞",
            font_size=44,
            color=ACCENT_COLOR,
            weight=BOLD
        ).move_to([0, -5.6, 0])
        conclusion.scale_to_fit_width(config.frame_width * 0.85)

        self.play(FadeIn(conclusion, shift=UP * 0.3))
        self.wait(2.5)

        # --- Closing Outro Card ---
        self.play(FadeOut(Group(*[m for m in self.mobjects if m != watermark])))

        closing_bg = Rectangle(
            width=config.frame_width,
            height=config.frame_height,
            fill_color=MAIN_COLOR,
            fill_opacity=1
        ).move_to(ORIGIN)

        closing_text = VGroup(
            Text("Made by Mugambi Ndwiga", font_size=46, weight=BOLD),
            Text("@craftsandengineering", font_size=32, color=GRAY_A, weight=MEDIUM)
        ).arrange(DOWN, buff=0.4).move_to(ORIGIN)

        self.play(FadeIn(closing_bg))
        self.play(Write(closing_text))
        self.wait(2)

Manim Community v0.20.1

In [ ]:
%%manim -v WARNING -r 1080,1920 --fps 30 --progress_bar None KochCurveAnimation

"""
Author: Mugambi Ndwiga
Aesthetic: Paper & Bleeding Fountain Pen Sketchbook Layout
Optimization: Prominent/Maximized Graphic, Highly Detailed Iterations (Level 5)
"""

from manim import *
import numpy as np

class KochCurveAnimation(Scene):
    def construct(self):
        # --- Force 9:16 Aspect Ratio Coordinate Space ---
        config.frame_width = 9
        config.frame_height = 16

        # --- Parchment & Fountain Pen Palette ---
        PAPER_BG = "#f4f1ea"        # Warm, textured sketchbook paper tone
        INK_MAIN = "#1c2d42"        # Deep Prussian Blue ink
        INK_ACCENT = "#8b1e2f"      # Bleeding Burgundy ink for active changes
        INK_MUTED = "#5c6b73"       # Soft graphite grey for secondary text

        self.camera.background_color = PAPER_BG

        # --- Hand-drawn Typography Configuration ---
        HANDWRITING_FONT = "Ink Free"
        HANDWRITTEN_MATH = TexFontTemplates.french_cursive

        # --- Watermark (Elegant margin signature) ---
        watermark = Text(
            "© Mugambi Ndwiga / @craftsandengineering",
            font=HANDWRITING_FONT,
            font_size=18,
            color=INK_MUTED,
            fill_opacity=0.6,
            warn_missing_font=False
        ).move_to([0, -7.6, 0])
        self.add(watermark)

        # --- Zone 1: Title Block (Shifted Higher to Free Up Central Space) ---
        title_main = Text("INFINITE PERIMETER", font=HANDWRITING_FONT, weight=BOLD, font_size=46, color=INK_MAIN, warn_missing_font=False)
        title_main.scale_to_fit_width(config.frame_width * 0.85)
        title_sub = Text("The Koch Curve", font=HANDWRITING_FONT, font_size=34, color=INK_MUTED, warn_missing_font=False)
        title = VGroup(title_main, title_sub).arrange(DOWN, buff=0.25).move_to([0, 6.3, 0])

        # --- Zone 2: Hand-written Static Formula Panel ---
        info_panel = MathTex(
            r"L_n = L_0 \cdot \left(\frac{4}{3}\right)^n",
            font_size=46,
            color=INK_MAIN,
            tex_template=HANDWRITTEN_MATH
        ).move_to([0, 4.3, 0])

        # --- Zone 3: Maximized Koch Curve Space ---
        # Baseline set wide (7.6 units out of 9) and dropped lower to allow massive vertical growth
        start_pt = np.array([-3.8, -1.5, 0])
        end_pt = np.array([3.8, -1.5, 0])
        curve = Line(start_pt, end_pt, color=INK_MAIN, stroke_width=6.0, stroke_opacity=0.95)

        # --- Geometry Logic ---
        def get_koch_points(points, order):
            for _ in range(order):
                new_points = []
                for i in range(len(points) - 1):
                    p1 = points[i]
                    p2 = points[i+1]
                    v = p2 - p1
                    a = p1 + v / 3
                    b = p1 + 2 * v / 3
                    rotation_matrix = np.array([
                        [np.cos(PI/3), -np.sin(PI/3), 0],
                        [np.sin(PI/3), np.cos(PI/3), 0],
                        [0, 0, 1]
                    ])
                    peak = a + np.dot(rotation_matrix, v / 3)
                    new_points.extend([p1, a, peak, b])
                new_points.append(points[-1])
                points = new_points
            return points

        # --- Narrative Flow ---
        self.play(Write(title), run_time=1.4)
        self.play(FadeIn(info_panel, shift=DOWN * 0.2))
        self.play(Create(curve))
        self.wait(0.6)

        # --- Zone 4: High-Iteration Dynamic Ink Transformations ---
        displayed_length = None

        # Increased to 5 deep iterations for complex fractal detail
        for n in range(1, 6):
            new_pts = get_koch_points([start_pt, end_pt], n)

            # Stroke width dynamically scales down as complexity builds to prevent bleeding blots
            current_stroke = max(1.8, 6.5 - (n * 0.9))
            new_curve = VMobject(color=INK_MAIN, stroke_width=current_stroke, stroke_opacity=0.95)
            new_curve.set_points_as_corners(new_pts)

            # Repositioned lower tracker zone to clear the expanding fractal base
            next_length = MathTex(
                f"L_{n} = {round((4/3)**n, 2)} \\cdot L_0",
                font_size=42,
                color=INK_ACCENT,
                tex_template=HANDWRITTEN_MATH
            ).move_to([0, -4.3, 0])

            if displayed_length is None:
                self.play(
                    Transform(curve, new_curve),
                    FadeIn(next_length, shift=UP * 0.2),
                    run_time=1.6
                )
                displayed_length = next_length
            else:
                self.play(
                    Transform(curve, new_curve),
                    Transform(displayed_length, next_length),
                    run_time=1.5
                )
            self.wait(0.8)

        # --- Zone 5: Lower Final Statement ---
        conclusion = Text(
            "As n → ∞, Length → ∞",
            font=HANDWRITING_FONT,
            font_size=42,
            color=INK_ACCENT,
            weight=BOLD,
            warn_missing_font=False
        ).move_to([0, -6.1, 0])
        conclusion.scale_to_fit_width(config.frame_width * 0.85)

        self.play(FadeIn(conclusion, shift=UP * 0.3))
        self.wait(2.5)

        # --- Outro Title Card (Consistent Sketchbook Branding) ---
        self.play(FadeOut(Group(*[m for m in self.mobjects if m != watermark])))

        closing_text = VGroup(
            Text("Made by Mugambi Ndwiga", font=HANDWRITING_FONT, font_size=44, weight=BOLD, color=INK_MAIN, warn_missing_font=False),
            Text("@craftsandengineering", font=HANDWRITING_FONT, font_size=32, color=INK_ACCENT, warn_missing_font=False)
        ).arrange(DOWN, buff=0.4).move_to(ORIGIN)

        self.play(Write(closing_text), run_time=1.8)
        self.wait(2.5)

Manim Community v0.20.1